# Step by step through the Remora algorithm for signal to sequence alignment

I adapted this from the Remora source code. I changed some variable names and got rid of the parent classes to make it better readable. After executing the code I compare it to the original Remora code directly to confirm that the results are the same.


The following functions are used to calculate the alignments (Structure corresponds to Remora implementation):
```
from_pod5_and_alignment
|
|--add_alignment
|    |
|    |-- trim_signal                
                (trim the signal based on split read tags (sp, ns, ts))
|    |-- revcomp                    
                (calculate the reverse complement if the mapping is reverse)
|    |-- adjust_move_table          
                (Adjust the query to signal alignment in case of reversed signal)
|
|--compute_ref_to_signal
    |
    |--compute_ref_to_signal_inner
        |
        |-- make_sequence_coordinate_mapping    
        |       (Set up the knots used to infer the ref to signal alignment)
        |-- map_ref_to_signal                   
                (Calculate the ref to signal alignment from the knots)
```

## Global variables to access some variables inside of functions

In [1]:
import pod5
from remora import io
from pathlib import Path
import numpy as np
test_data_root = Path("../example_data")
pod5_dr = pod5.DatasetReader(test_data_root)
bam_fh = io.ReadIndexedBam(test_data_root / "can_mappings.bam")

read_id = "6e37823a-9398-4be8-b111-65cab029f4e0"
pod5_read = pod5_dr.get_read(read_id)
bam_read = bam_fh.get_first_alignment(read_id)

Indexing BAM by parent read id: 14 Reads [00:00, 15575.66 Reads/s]


## Functions for query- and ref-to-signal alignment

### from_pod5_and_alignment

In [2]:
def from_pod5_and_alignment(pod5_read_record: pod5.reader.ReadRecord, alignment_record, reverse_signal=False):
    
    signal = pod5_read_record.signal
    if reverse_signal:
        signal = signal[::-1]
    # calibration offset and scale apply normalization as:
    #     y=(x+a)/b
    # while io.Read stores shfit and scale as:
    #     y=c*(x-d)
    # In other words, as shift=mean and scale=std. dev.
    alignments = add_alignment(
        signal,
        alignment_record,
        reverse_signal,
    )
    return alignments


### add_alignment

In [3]:
def add_alignment(
    signal,
    alignment_record,
    reverse_signal=False
):
    """Add alignment to read object

    Args:
        alignment_record (pysam.AlignedSegment)
        parse_ref_align (bool): Should reference alignment be parsed
        reverse_signal (bool): Does this read derive from 3' to 5' signal
            (RNA reads)
        pa_scaling (tuple): picoamp to zero-centered picoamp shift and scale
    """
    if (
        alignment_record.reference_name is None
        and alignment_record.is_reverse
    ):
        raise Exception("Unmapped reads cannot map to reverse strand.")

    tags = dict(alignment_record.tags)

    trim_tags = dict(
        (tag, tags.get(tag, dv))
        for tag, dv in (("sp", 0), ("ts", 0), ("ns", None))
    )

    signal = trim_signal(signal, trim_tags, reverse_signal)

    global DEBUG_SIGNAL
    DEBUG_SIGNAL = signal

    parent_read_id = tags.get("pi", None)
    if parent_read_id is None:
        if alignment_record.query_name != read_id:
            raise Exception("Read IDs mismatch")
    else:
        if parent_read_id != read_id:
            raise Exception("Split read IDs mismatch")
        child_read_id = alignment_record.query_name

    seq = alignment_record.query_sequence
    if alignment_record.is_reverse:
        seq = revcomp(seq)
    try:
        stride = tags["mv"][0]
        mv_table = np.array(tags["mv"][1:])
        query_to_signal = np.nonzero(mv_table)[0] * stride
        if signal is not None:
            query_to_signal = adjust_move_table(query_to_signal, 
                                                len(signal), 
                                                len(seq), 
                                                len(mv_table), 
                                                stride, 
                                                reverse_signal=reverse_signal)
    except KeyError:
        raise Exception(f"Move table not found")

    ref_seq = alignment_record.get_reference_sequence().upper()
    cigar = alignment_record.cigartuples
    if alignment_record.is_reverse:
        if ref_seq is not None:
            ref_seq = revcomp(ref_seq)
        cigar = cigar[::-1]
    ref_to_signal = compute_ref_to_signal(query_to_signal, cigar)

    return query_to_signal, ref_to_signal

def trim_signal(signal, trim_tags, reverse_signal):
    if reverse_signal:
        signal = signal[::-1]
    # trim for split read sp tag
    signal = signal[trim_tags["sp"] :]
    # trim for start and end read trimming
    ns = trim_tags["ns"]
    if ns is None:
        ns = signal.size
    signal = signal[trim_tags["ts"] : ns]
    if reverse_signal:
        signal = signal[::-1]

    return signal

def revcomp(seq):
    """Convert seq to its complement sequence and reverse the sequence.
    Handles IUPAC ambiguous bases.
    """
    COMP_BASES = dict(zip(map(ord, "ACGTBVDHKMRY"), map(ord, "TGCAVBHDMKYR")))

    return seq.upper().translate(COMP_BASES)[::-1]

def adjust_move_table(query_to_signal, 
                      sig_len, 
                      seq_len,
                      mv_table_size,
                      stride,
                      reverse_signal=False, 
                      check=True):
    # add last point to query_to_signal
    query_to_signal = np.concatenate(
        [query_to_signal, [sig_len]]
    )
    if reverse_signal:
        query_to_signal = sig_len - query_to_signal[::-1]
    if check:
        if query_to_signal.size - 1 != seq_len:
            raise Exception("Move table discordant with basecalls")
        if mv_table_size != sig_len // stride:
            raise Exception("Move table discordant with signal")
    return query_to_signal


### compute_ref_to_signal

In [4]:
def compute_ref_to_signal(query_to_signal, cigar):
    ref_to_signal = compute_ref_to_signal_inner(
        query_to_signal=query_to_signal,
        cigar=cigar,
    )
    return ref_to_signal

def compute_ref_to_signal_inner(query_to_signal, cigar):
    ref_to_read_knots = make_sequence_coordinate_mapping(cigar)
    return map_ref_to_signal(
        query_to_signal=query_to_signal, ref_to_query_knots=ref_to_read_knots
    )

def make_sequence_coordinate_mapping(cigar):
    """Maps an element in reference to every element in basecalls using
    alignment in `cigar`.

    Args:
        cigar (list): "cigartuples" representing alignment

    Returns:
        array shape (ref_len,). [x_0, x_1, ..., x_(ref_len)]
            such that read_seq[x_i] <> ref_seq[i]. Note that ref_len is derived
            from the cigar input.
    """
    MATCH_OPS = np.array(
        [True, False, False, False, False, False, False, True, True]
    )
    MATCH_OPS_SET = set(np.where(MATCH_OPS)[0])

    QUERY_OPS = np.array([True, True, False, False, True, False, False, True, True])
    REF_OPS = np.array([True, False, True, True, False, False, False, True, True])


    while len(cigar) > 0 and cigar[-1][0] not in MATCH_OPS_SET:
        cigar = cigar[:-1]
    if len(cigar) == 0:
        raise Exception("No match operations found in alignment cigar")
    
    global DEBUG_CIGAR
    DEBUG_CIGAR = cigar

    ops, lens = map(np.array, zip(*cigar))
    if ops.min() < 0 or ops.max() > 8:
        raise Exception("Invalid cigar op(s)")
    if lens.min() < 0:
        raise Exception("Cigar lengths may not be negative")

    is_match = MATCH_OPS[ops]
    match_counts = lens[is_match]
    offsets = np.array([match_counts, np.ones_like(match_counts)])

    # TODO remove knots around ambiguous indels (e.g. left justified HPs)
    # note this requires the ref and query sequences
    ref_knots = np.cumsum(np.where(REF_OPS[ops], lens, 0))
    ref_knots = np.concatenate(
        [[0], (ref_knots[is_match] - offsets).T.flatten(), [ref_knots[-1]]]
    )
    query_knots = np.cumsum(np.where(QUERY_OPS[ops], lens, 0))
    query_knots = np.concatenate(
        [[0], (query_knots[is_match] - offsets).T.flatten(), [query_knots[-1]]]
    )
    knots = np.interp(np.arange(ref_knots[-1] + 1), ref_knots, query_knots)

    global DEBUG_KNOTS, DEBUG_REF_KNOTS, DEBUG_QUERY_KNOTS
    DEBUG_KNOTS = knots
    DEBUG_REF_KNOTS = ref_knots
    DEBUG_QUERY_KNOTS = query_knots

    return knots

def map_ref_to_signal(*, query_to_signal, ref_to_query_knots):
    """Compute interpolated mapping from reference, through query alignment to
    signal coordinates

    Args:
        query_to_signal (np.array): Query to signal coordinate mapping
        ref_to_query_knots (np.array): Reference to query coordinate mapping
    """
    return np.floor(
        np.interp(
            ref_to_query_knots,
            np.arange(query_to_signal.size),
            query_to_signal,
        )
    ).astype(int)



## Perform alignment

In [8]:
query_to_signal, ref_to_signal = from_pod5_and_alignment(pod5_read, bam_read)
print(query_to_signal)
print(ref_to_signal)

[    0    45    50 ... 83205 83225 83234]
[ 1060  1065  1080 ... 82780 82805 82815]


In [9]:
read = io.Read.from_pod5_and_alignment(pod5_read, bam_read)
print(read.query_to_signal)
print(read.ref_to_signal)

[    0    45    50 ... 83205 83225 83234]
[ 1060  1065  1080 ... 82780 82805 82815]


In [10]:
print(query_to_signal == read.query_to_signal, "Number of unequal values:", query_to_signal.size - np.count_nonzero((query_to_signal == read.query_to_signal)))
print(ref_to_signal == read.ref_to_signal, "Number of unequal values:", ref_to_signal.size - np.count_nonzero((ref_to_signal == read.ref_to_signal)))

[ True  True  True ...  True  True  True] Number of unequal values: 0
[ True  True  True ...  True  True  True] Number of unequal values: 0


## Inspecting reference to signal alignment in more detail

Need the following variables for this:
- signal
- query_to_signal
- cigar
- ref_knots
- query_knots
- knots (interpolated)
- ref_to_signal

In [11]:
signal = DEBUG_SIGNAL
cigar = DEBUG_CIGAR
ref_knots = DEBUG_REF_KNOTS
query_knots = DEBUG_QUERY_KNOTS
knots = DEBUG_KNOTS

Comparing the signal from the function to the initial signal: The initial signal is 10 measurements longer. This corresponds to the 

In [17]:
print(pod5_read.signal, len(pod5_read.signal))
print(signal, len(signal))
print(f"Tags:  ts={bam_read.get_tag("ts")}, ns={bam_read.get_tag("ns")} -->", pod5_read.signal[bam_read.get_tag("ts"):bam_read.get_tag("ns")])

[ 976 1002  956 ...  779  745  733] 83244
[ 987  990 1011 ...  779  745  733] 83234
Tags:  ts=10, ns=83244 --> [ 987  990 1011 ...  779  745  733]


### Looking into the CIGAR tuples

Comparing the CIGAR tuples from the function with the original ones: CIGAR tuples are reversed (because read is reverse mapped) and the now last CIGAR element is removed (4=S -> Soft-Clip) (WHY IS IT KEPT IN THE BEGINNING THOUGH?)

In [18]:
print(len(bam_read.cigartuples), bam_read.cigartuples)
print(len(cigar), cigar)
print("Is read reverse?", bam_read.is_reverse)

65 [(4, 36), (0, 655), (2, 1), (0, 23), (2, 1), (0, 664), (2, 1), (0, 517), (1, 1), (0, 3), (1, 1), (0, 166), (2, 4), (0, 107), (2, 1), (0, 169), (1, 6), (0, 206), (2, 1), (0, 39), (2, 2), (0, 4), (1, 1), (0, 241), (2, 1), (0, 4), (2, 2), (0, 502), (1, 2), (0, 122), (1, 1), (0, 2), (2, 1), (0, 59), (2, 1), (0, 60), (2, 1), (0, 13), (2, 1), (0, 19), (1, 1), (0, 168), (2, 2), (0, 33), (1, 2), (0, 486), (2, 1), (0, 496), (1, 3), (0, 462), (1, 1), (0, 31), (2, 1), (0, 14), (1, 2), (0, 518), (1, 1), (0, 2), (1, 1), (0, 7), (1, 1), (0, 910), (2, 1), (0, 54), (4, 86)]
64 [(4, 86), (0, 54), (2, 1), (0, 910), (1, 1), (0, 7), (1, 1), (0, 2), (1, 1), (0, 518), (1, 2), (0, 14), (2, 1), (0, 31), (1, 1), (0, 462), (1, 3), (0, 496), (2, 1), (0, 486), (1, 2), (0, 33), (2, 2), (0, 168), (1, 1), (0, 19), (2, 1), (0, 13), (2, 1), (0, 60), (2, 1), (0, 59), (2, 1), (0, 2), (1, 1), (0, 122), (1, 2), (0, 502), (2, 2), (0, 4), (2, 1), (0, 241), (1, 1), (0, 4), (2, 2), (0, 39), (2, 1), (0, 206), (1, 6), (0, 16

1595 shows how deletions are handled. The range that was allocated to the one base (T) in the query to signal alignment gets split evenly to fit all additional bases (T,G):

In [ ]:
import plotly.graph_objects as go

def find_start_end_idx(f: int, t: int, alignment: np.ndarray) -> tuple[int,int]:
    start = 0
    end = alignment.size

    start_found = False
    for idx,val in enumerate(alignment):
        if val >= f and not start_found:
            start = max(idx-1, 0)
            start_found = True

        if val >= t:
            end = min(query_to_signal.size, idx+1)
            break
    return start, end

def plot_from_to(f: int, t: int) -> go.Figure:
    t = min(t, signal.size)

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=np.arange(f,t),
            y=signal[f:t],
        )
    )

    start, end = find_start_end_idx(f,t, query_to_signal)
    qts_subset = query_to_signal[start:end]
    q_subset = bam_read.seq[start:end]
    for i in range(len(qts_subset)-1):
        fig.add_vline(x=qts_subset[i], line_color="green", line_dash="solid")
        fig.add_annotation(x=(qts_subset[i]+qts_subset[i+1])/2, y=1, 
                        text=q_subset[i],
                        showarrow=False,
                        bgcolor="green")


    start, end = find_start_end_idx(f,t, ref_to_signal)
    rts_subset = ref_to_signal[start:end]
    r_subset = bam_read.query_alignment_sequence[start:end]
    for i in range(len(rts_subset)-1):
        fig.add_vline(x=rts_subset[i], line_color="red", line_dash="dot")
        fig.add_annotation(x=(rts_subset[i]+rts_subset[i+1])/2, y=100, 
                        text=r_subset[i],
                        showarrow=False,
                        bgcolor="red")
    fig.layout.yaxis.fixedrange = True
    return fig



In [186]:
plot_from_to(1500,1700)


### Looking into the knots in more detail

Making the code more readable:

In [129]:
def make_sequence_coordinate_mapping_clearer(cigar):
    """Maps an element in reference to every element in basecalls using
    alignment in `cigar`.

    Args:
        cigar (list): "cigartuples" representing alignment

    Returns:
        array shape (ref_len,). [x_0, x_1, ..., x_(ref_len)]
            such that read_seq[x_i] <> ref_seq[i]. Note that ref_len is derived
            from the cigar input.
    """
    MATCH_OPS = np.array(
        [True, False, False, False, False, False, False, True, True]
    )
    MATCH_OPS_SET = set(np.where(MATCH_OPS)[0])

    QUERY_OPS = np.array([True, True, False, False, True, False, False, True, True])
    REF_OPS = np.array([True, False, True, True, False, False, False, True, True])


    while len(cigar) > 0 and cigar[-1][0] not in MATCH_OPS_SET:
        cigar = cigar[:-1]
    if len(cigar) == 0:
        raise Exception("No match operations found in alignment cigar")
    
    global DEBUG_CIGAR_clear
    DEBUG_CIGAR_clear = cigar

    ops, lens = map(np.array, zip(*cigar))
    if ops.min() < 0 or ops.max() > 8:
        raise Exception("Invalid cigar op(s)")
    if lens.min() < 0:
        raise Exception("Cigar lengths may not be negative")

    # is_match = MATCH_OPS[ops]
    is_match = []
    for op in ops:
        if op in [0,7,8]:
            is_match.append(True)
        else:
            is_match.append(False)

    # match_counts = lens[is_match]
    match_counts = []
    for l, is_m in zip(lens, is_match):
        if is_m:
            match_counts.append(l)

    # offsets = np.array([match_counts, np.ones_like(match_counts)])
    offsets = []
    for i in match_counts:
        offsets.append(i)

    # ref_knots = np.cumsum(np.where(REF_OPS[ops], lens, 0))
    ref_knots = []
    current_sum = 0
    for op, l in zip(ops, lens):
        if op not in [0,2,3,7,8]:
            l = 0
        current_sum += l
        ref_knots.append(current_sum)

    # ref_knots = np.concatenate(
    #     [[0], (ref_knots[is_match] - offsets).T.flatten(), [ref_knots[-1]]]
    # )
    ref_knots_match = []
    for op, el in zip(ops, ref_knots):
        if op in [0,7,8]:
            ref_knots_match.append(el)

    ref_knots_new = [0]
    for el, offset in zip(ref_knots_match, offsets):
        ref_knots_new.append(el-offset)
        ref_knots_new.append(el-1)
    ref_knots_new.append(ref_knots[-1])
    ref_knots = ref_knots_new
    
    # query_knots = np.cumsum(np.where(QUERY_OPS[ops], lens, 0))
    query_knots = []
    current_sum = 0
    for op,l in zip(ops,lens):
        if op not in [0,1,4,7,8]:
            l = 0
        current_sum += l
        query_knots.append(current_sum)


    # query_knots = np.concatenate(
        # [[0], (query_knots[is_match] - offsets).T.flatten(), [query_knots[-1]]]
    # )
    query_knots_match = []
    for op, el in zip(ops, query_knots):
        if op in [0,7,8]:
            query_knots_match.append(el)

    query_knots_new = [0]
    for el, offset in zip(query_knots_match, offsets):
        query_knots_new.append(el-offset)
        query_knots_new.append(el-1)
    query_knots_new.append(query_knots[-1])
    query_knots = query_knots_new

    knots = np.interp(np.arange(ref_knots[-1] + 1), np.array(ref_knots), np.array(query_knots))

    global DEBUG_KNOTS_clear, DEBUG_REF_KNOTS_clear, DEBUG_QUERY_KNOTS_clear
    DEBUG_KNOTS_clear = knots
    DEBUG_REF_KNOTS_clear = ref_knots
    DEBUG_QUERY_KNOTS_clear = query_knots

    return knots


In [189]:
def make_sequence_coordinate_mapping_clearest(cigar):
    """Maps an element in reference to every element in basecalls using
    alignment in `cigar`.

    Args:
        cigar (list): "cigartuples" representing alignment

    Returns:
        array shape (ref_len,). [x_0, x_1, ..., x_(ref_len)]
            such that read_seq[x_i] <> ref_seq[i]. Note that ref_len is derived
            from the cigar input.
    """
    def is_match_op(op) -> bool:
        return op in [0,7,8]
    
    def is_ref_op(op) -> bool:
        return op in [0,2,3,7,8]
    
    def is_query_op(op) -> bool:
        return op in [0,1,4,7,8]

    while len(cigar) > 0 and not is_match_op(cigar[-1][0]):
        cigar = cigar[:-1]
    if len(cigar) == 0:
        raise Exception("No match operations found in alignment cigar")
    
    global DEBUG_CIGAR_clear_v2
    DEBUG_CIGAR_clear_v2 = cigar

    ops, lens = map(np.array, zip(*cigar))
    if ops.min() < 0 or ops.max() > 8:
        raise Exception("Invalid cigar op(s)")
    if lens.min() < 0:
        raise Exception("Cigar lengths may not be negative")

    ref_knots = [0]
    current_ref_knot = 0
    query_knots = [0]
    current_query_knot = 0

    for op, l in cigar:
        l_ref_knot = l if is_ref_op(op) else 0
        start_r = current_ref_knot
        end_r = current_ref_knot + l_ref_knot

        l_query_knot = l if is_query_op(op) else 0
        start_q = current_query_knot
        end_q = current_query_knot + l_query_knot

        if is_match_op(op):
            ref_knots.append(start_r)
            ref_knots.append(end_r - 1)
            
            query_knots.append(start_q)
            query_knots.append(end_q - 1)

        current_ref_knot = end_r
        current_query_knot = end_q

    ref_knots.append(current_ref_knot)
    query_knots.append(current_query_knot)

    print(ref_knots, query_knots)

    knots = np.interp(np.arange(ref_knots[-1] + 1), np.array(ref_knots), np.array(query_knots))

    global DEBUG_KNOTS_clear_v2, DEBUG_REF_KNOTS_clear_v2, DEBUG_QUERY_KNOTS_clear_v2
    DEBUG_KNOTS_clear_v2 = knots
    DEBUG_REF_KNOTS_clear_v2 = ref_knots
    DEBUG_QUERY_KNOTS_clear_v2 = query_knots

    return knots


In [190]:
make_sequence_coordinate_mapping_clearest([(7,1), (2,1), (7,3), (8,1), (7,1)])

[0, 0, 0, 2, 4, 5, 5, 6, 6, 7] [0, 0, 0, 1, 3, 4, 4, 5, 5, 6]


array([0. , 0.5, 1. , 2. , 3. , 4. , 5. , 6. ])

- For each cigar operation, it is checked if is is an operation that steps along the reference/query sequence (M,D,N,=,X)/(M,I,S,=,X). 
- If the operation is also a match operation (M,=,X) both the start index (the cumulative sum of all reference advancing operations) and the end index (start index + length of the operation) - 1 gets appended to the reference/query knot list. 
- Afterwards the reference/query start index gets updated for the next operation (stays the same if it wasn't a reference/query advancing operation). 

Is the result exactly the same?


In [131]:
make_sequence_coordinate_mapping(bam_read.cigar) == make_sequence_coordinate_mapping_clearer(bam_read.cigar)


array([ True,  True,  True, ...,  True,  True,  True])

In [132]:
make_sequence_coordinate_mapping(bam_read.cigar) == make_sequence_coordinate_mapping_clearest(bam_read.cigar)


array([ True,  True,  True, ...,  True,  True,  True])

Yes!

What shape do the ref and query knots have?

In [143]:
print(f"Length of ref knots: {ref_knots.size}\nLength of query knots: {query_knots.size}\nNumber of match operations: {len([i for i in cigar if i[0] in [0,7,8]])}")

Length of ref knots: 66
Length of query knots: 66
Number of match operations: 32


Length of ref and query knots is the same. It corresponds to 2x the number of match operations.

` #knots = 1 + 2 x #match_ops + 1 `

This means that the knots correspond to start and end intervals in the alignment, where both ref and query advances.

In [150]:
for i in range(len(ref_knots)-1):
    print(f"Start: {ref_knots[i]} - {query_knots[i]}\t| End: {ref_knots[i+1]} - {query_knots[i+1]}")

Start: 0 - 0	| End: 0 - 86
Start: 0 - 86	| End: 53 - 139
Start: 53 - 139	| End: 55 - 140
Start: 55 - 140	| End: 964 - 1049
Start: 964 - 1049	| End: 965 - 1051
Start: 965 - 1051	| End: 971 - 1057
Start: 971 - 1057	| End: 972 - 1059
Start: 972 - 1059	| End: 973 - 1060
Start: 973 - 1060	| End: 974 - 1062
Start: 974 - 1062	| End: 1491 - 1579
Start: 1491 - 1579	| End: 1492 - 1582
Start: 1492 - 1582	| End: 1505 - 1595
Start: 1505 - 1595	| End: 1507 - 1596
Start: 1507 - 1596	| End: 1537 - 1626
Start: 1537 - 1626	| End: 1538 - 1628
Start: 1538 - 1628	| End: 1999 - 2089
Start: 1999 - 2089	| End: 2000 - 2093
Start: 2000 - 2093	| End: 2495 - 2588
Start: 2495 - 2588	| End: 2497 - 2589
Start: 2497 - 2589	| End: 2982 - 3074
Start: 2982 - 3074	| End: 2983 - 3077
Start: 2983 - 3077	| End: 3015 - 3109
Start: 3015 - 3109	| End: 3018 - 3110
Start: 3018 - 3110	| End: 3185 - 3277
Start: 3185 - 3277	| End: 3186 - 3279
Start: 3186 - 3279	| End: 3204 - 3297
Start: 3204 - 3297	| End: 3206 - 3298
Start: 3206 - 

In [165]:
import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=ref_knots, y=query_knots, mode='lines+markers')
)
fig.update_layout(
    xaxis_title="Reference Knots",
    yaxis_title="Query Knots",
    width=650,
    height=600
)

fig.show()

In [187]:
ref_knots

array([   0,    0,   53,   55,  964,  965,  971,  972,  973,  974, 1491,
       1492, 1505, 1507, 1537, 1538, 1999, 2000, 2495, 2497, 2982, 2983,
       3015, 3018, 3185, 3186, 3204, 3206, 3218, 3220, 3279, 3281, 3339,
       3341, 3342, 3343, 3464, 3465, 3966, 3969, 3972, 3974, 4214, 4215,
       4218, 4221, 4259, 4261, 4466, 4467, 4635, 4637, 4743, 4748, 4913,
       4914, 4916, 4917, 5433, 5435, 6098, 6100, 6122, 6124, 6778, 6779])

In [188]:
query_knots

array([   0,   86,  139,  140, 1049, 1051, 1057, 1059, 1060, 1062, 1579,
       1582, 1595, 1596, 1626, 1628, 2089, 2093, 2588, 2589, 3074, 3077,
       3109, 3110, 3277, 3279, 3297, 3298, 3310, 3311, 3370, 3371, 3429,
       3430, 3431, 3433, 3554, 3557, 4058, 4059, 4062, 4063, 4303, 4305,
       4308, 4309, 4347, 4348, 4553, 4560, 4728, 4729, 4835, 4836, 5001,
       5003, 5005, 5007, 5523, 5524, 6187, 6188, 6210, 6211, 6865, 6866])

The reference and query knots make up a curve where each index of the reference sequence corresponds to a index in the query sequence.



When interpolating x-values 0 to the length of the reference sequence, we can extract the indices of the corresponding intervals on the query sequence.

Conveniently we can translate indices of the 

In [182]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(x=np.arange(query_to_signal.size), y=query_to_signal, mode='lines')
)

fig.update_layout(
    xaxis_title="Query to signal indices",
    yaxis_title="Query to signal",
    width=650,
    height=600
)

fig.show()